# Preprocessing Dataset ABSA Hotel Santika

Notebook ini melakukan preprocessing pada `dataset_absa_santika.csv` sebelum tahap labeling/anotasi.

**Input dataset (sudah dipersiapkan oleh notebook merge):**
- `text_review` — teks final, sudah ditranslate ke Indonesia jika sebelumnya bahasa lain
- `text_review_original` — teks asli sebelum translate (untuk transparansi/audit)
- `original_language` — kode bahasa asli (id, en, ja, ko, dll)

**Referensi best practice:**
- SemEval 2014-2016 ABSA shared task guidelines
- IndoBERT preprocessing pipeline (Koto et al., 2020)
- ABSA hotel domain preprocessing (Pontiki et al., 2016)

**Tahapan:**
1. Load dataset
2. Analisis kualitas data (panjang teks, duplikat, bahasa, tanggal)
3. Hapus exact duplicates
4. Filter review terlalu pendek (< 20 karakter)
5. Normalisasi teks (emoji, spasi, karakter khusus)
6. Case folding (lowercasing)
7. Normalisasi slang/singkatan Indonesia
8. Normalisasi karakter berulang
9. Bersihkan tanda baca berlebihan
10. Final cleaning & near-duplicate check
11. Export dataset bersih (mempertahankan kolom `text_review_original` untuk audit)

**PENTING untuk ABSA:**
- TIDAK menghapus stopwords (kata negasi seperti 'tidak', 'bukan', 'kurang' sangat krusial)
- TIDAK melakukan stemming agresif (mempertahankan makna kata untuk deteksi aspek)
- Mempertahankan tanda baca penting (koma, titik untuk sentence splitting saat labeling)

---
## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import re
import os
from collections import Counter

print('Libraries loaded.')

---
## 2. Load Dataset

In [ ]:
BASE_DIR = '/content/drive/MyDrive/Scrap_Review_Santika'
INPUT_FILE = 'dataset_absa_santika.csv'
OUTPUT_FILE = 'dataset_absa_santika_clean.csv'

df = pd.read_csv(os.path.join(BASE_DIR, INPUT_FILE), encoding='utf-8-sig')

print(f'Dataset loaded: {len(df):,} review')
print(f'Kolom: {df.columns.tolist()}')
display(df.head())

# Simpan jumlah awal untuk tracking
initial_count = len(df)

---
## 3. Analisis Kualitas Data (Pre-Cleaning)

In [ ]:
print('=== ANALISIS KUALITAS DATA ===')
print(f'Total review: {len(df):,}')

# Text length
df['_text_len'] = df['text_review'].astype(str).str.len()
print(f'\nPanjang teks (text_review):')
print(f'  Min: {df["_text_len"].min()}')
print(f'  Max: {df["_text_len"].max()}')
print(f'  Mean: {df["_text_len"].mean():.0f}')
print(f'  Median: {df["_text_len"].median():.0f}')

# Duplicates
n_dupes = df['text_review'].duplicated().sum()
print(f'\nDuplikat (text_review): {n_dupes:,} review')

# Short reviews
n_short = (df['_text_len'] < 20).sum()
print(f'Review pendek (<20 char): {n_short:,}')

# Language distribution
if 'original_language' in df.columns:
    print(f'\nDistribusi bahasa asli:')
    print(df['original_language'].value_counts().to_string())

# Berapa review yang ditranslate
if 'text_review_original' in df.columns:
    n_translated = (df['text_review'] != df['text_review_original']).sum()
    print(f'\nReview yang sudah ditranslate ke Indonesia: {n_translated:,}')

# Rentang tanggal
if 'date' in df.columns:
    print(f'\nRentang tanggal:')
    print(f'  Terlama: {df["date"].min()}')
    print(f'  Terbaru: {df["date"].max()}')
    # Distribusi per tahun
    df['_year'] = pd.to_datetime(df['date'], errors='coerce').dt.year
    print(f'\nDistribusi per tahun:')
    print(df['_year'].value_counts().sort_index().to_string())
    df.drop(columns=['_year'], inplace=True)

# Emoji count
emoji_pattern = re.compile('[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF\U00002702-\U000027B0\U0001f900-\U0001f9FF\U00002600-\U000026FF\u200d\ufe0f]')
n_emoji = df['text_review'].astype(str).apply(lambda x: bool(emoji_pattern.search(x))).sum()
print(f'\nReview dengan emoji: {n_emoji:,}')

df.drop(columns=['_text_len'], inplace=True)

---
## 4. Step 1: Hapus Duplikat Exact

Duplikat exact berasal dari scraping yang overlap atau paginasi berulang. Pertahankan kemunculan pertama saja.

In [ ]:
before = len(df)
df = df.drop_duplicates(subset='text_review', keep='first')
removed = before - len(df)
print(f'Hapus duplikat: {removed:,} review dihapus')
print(f'Sisa: {len(df):,} review')

---
## 5. Step 2: Filter Review Terlalu Pendek

Review < 20 karakter (contoh: "Ok", "Nice", "recommed") tidak memiliki cukup konteks untuk ABSA.
Tidak bisa diekstrak aspek maupun sentimen dengan reliable.

In [ ]:
MIN_LENGTH = 20

# Tampilkan sample yang akan dihapus
short = df[df['text_review'].astype(str).str.len() < MIN_LENGTH]
print(f'Review < {MIN_LENGTH} char: {len(short):,}')
print('\nSample yang dihapus:')
for _, row in short.head(10).iterrows():
    print(f'  [{row["platform"]}] "{row["text_review"]}"')

before = len(df)
df = df[df['text_review'].astype(str).str.len() >= MIN_LENGTH]
removed = before - len(df)
print(f'\nDihapus: {removed:,} review')
print(f'Sisa: {len(df):,} review')

---
## 6. Step 3: Normalisasi Teks

Pembersihan karakter tanpa menghilangkan makna:
- Hapus emoji (atau konversi ke teks)
- Normalisasi spasi ganda
- Hapus karakter non-printable
- Pertahankan tanda baca penting (. , ! ? - :)

**PENTING:** Tidak menghapus kata negasi dan tanda baca karena krusial untuk ABSA.

In [ ]:
# Mapping emoji ke teks sentimen (opsional, bisa dibuang juga)
EMOJI_SENTIMENT = {
    '😀': '', '😃': '', '😄': '', '😁': '', '😆': '',
    '😅': '', '🤣': '', '😂': '', '🙂': '', '😊': '',
    '😍': '', '🥰': '', '😘': '', '😗': '', '😙': '',
    '😚': '', '😋': '', '😛': '', '😜': '', '🤪': '',
    '😝': '', '🤗': '', '🤭': '', '🤫': '', '🤔': '',
    '😐': '', '😑': '', '😶': '', '😏': '', '😒': '',
    '🙄': '', '😬': '', '😮': '', '😯': '', '😲': '',
    '😳': '', '🥺': '', '😢': '', '😭': '', '😤': '',
    '😠': '', '😡': '', '🤬': '', '😈': '', '👿': '',
    '👍': '', '👎': '', '👏': '', '🙏': '',
    '❤️': '', '💯': '', '⭐': '', '🌟': '',
    '✅': '', '❌': '', '✨': '', '🔥': '',
    '🥴': '', '💪': '', '😎': '',
}


def normalize_text(text):
    """Normalisasi teks review untuk ABSA."""
    if pd.isna(text) or not isinstance(text, str):
        return ''

    # 1. Hapus emoji
    for emoji, replacement in EMOJI_SENTIMENT.items():
        text = text.replace(emoji, replacement)
    # Hapus sisa emoji yang tidak ada di mapping
    text = emoji_pattern.sub('', text)

    # 2. Hapus karakter non-printable & kontrol Unicode
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)

    # 3. Normalisasi tanda kutip & apostrof
    text = text.replace('\u201c', '"').replace('\u201d', '"')  # smart quotes
    text = text.replace('\u2018', "'").replace('\u2019', "'")  # smart apostrophe
    text = text.replace('\u2014', ' - ')  # em dash
    text = text.replace('\u2013', ' - ')  # en dash

    # 4. Normalisasi whitespace
    text = re.sub(r'\n+', ' ', text)  # newline -> spasi
    text = re.sub(r'\t+', ' ', text)  # tab -> spasi
    text = re.sub(r' {2,}', ' ', text)  # multiple spaces -> single

    # 5. Hapus spasi di awal/akhir
    text = text.strip()

    return text


# Apply
print('Menerapkan normalisasi teks...')
df['text_review'] = df['text_review'].apply(normalize_text)

# Hapus review yang jadi kosong setelah normalisasi
before = len(df)
df = df[df['text_review'].str.strip().astype(bool)]
df = df[df['text_review'].str.len() >= MIN_LENGTH]
removed = before - len(df)
print(f'Review kosong setelah normalisasi: {removed:,} dihapus')
print(f'Sisa: {len(df):,} review')

# Sample
print('\nSample setelah normalisasi:')
for _, row in df.head(5).iterrows():
    preview = row['text_review'][:100] + '...' if len(row['text_review']) > 100 else row['text_review']
    print(f'  [{row["platform"]}] {preview}')

---
## 7. Step 4: Case Folding (Lowercasing)

Untuk konsistensi dan mengurangi vocabulary size. Penting karena IndoBERT menggunakan uncased model.

In [ ]:
df['text_review'] = df['text_review'].str.lower()
print('Case folding selesai.')

# Sample
for _, row in df.head(3).iterrows():
    preview = row['text_review'][:100] + '...' if len(row['text_review']) > 100 else row['text_review']
    print(f'  {preview}')

---
## 8. Step 5: Normalisasi Slang & Singkatan Indonesia

Review informal mengandung banyak singkatan yang bisa mempengaruhi deteksi aspek.
Normalisasi ini membantu model dan annotator memahami teks dengan lebih konsisten.

In [ ]:
# Kamus normalisasi slang/singkatan Indonesia
SLANG_DICT = {
    # Singkatan umum
    'yg': 'yang',
    'dgn': 'dengan',
    'dg': 'dengan',
    'utk': 'untuk',
    'krn': 'karena',
    'tp': 'tapi',
    'tdk': 'tidak',
    'gak': 'tidak',
    'ga': 'tidak',
    'gk': 'tidak',
    'nggak': 'tidak',
    'enggak': 'tidak',
    'trs': 'terus',
    'trus': 'terus',
    'bgt': 'banget',
    'bngt': 'banget',
    'bkn': 'bukan',
    'blm': 'belum',
    'sdh': 'sudah',
    'udh': 'sudah',
    'udah': 'sudah',
    'lg': 'lagi',
    'lgi': 'lagi',
    'jg': 'juga',
    'jgn': 'jangan',
    'sm': 'sama',
    'dr': 'dari',
    'dlm': 'dalam',
    'dpt': 'dapat',
    'hrs': 'harus',
    'spy': 'supaya',
    'ttg': 'tentang',
    'org': 'orang',
    'sy': 'saya',
    'ak': 'aku',
    'kmu': 'kamu',
    'km': 'kamu',
    'bs': 'bisa',
    'cm': 'cuma',
    'kl': 'kalau',
    'klo': 'kalau',
    'kalo': 'kalau',
    'mkn': 'mungkin',
    'mgkn': 'mungkin',
    'emg': 'memang',
    'emang': 'memang',
    'bbrp': 'beberapa',
    'brg': 'barang',
    'bgmn': 'bagaimana',
    'gmn': 'gimana',
    'dmn': 'dimana',
    'kmr': 'kamar',
    'tmpt': 'tempat',
    'smua': 'semua',
    'bnr': 'benar',
    'bner': 'benar',
    'ckp': 'cukup',
    'scr': 'secara',
    'trm': 'terima',
    'trmksh': 'terima kasih',
    'thx': 'terima kasih',
    'thanks': 'terima kasih',
    'thank': 'terima kasih',
    'tq': 'terima kasih',
    'makasih': 'terima kasih',
    'mksh': 'terima kasih',
    'mks': 'terima kasih',
    # Slang ekspresif
    'mantap': 'bagus',
    'mantapp': 'bagus',
    'mantapppp': 'bagus',
    'mantul': 'bagus',
    'keren': 'bagus',
    'oke': 'baik',
    'okee': 'baik',
    'okeee': 'baik',
    'okeeee': 'baik',
    'okey': 'baik',
    'ok': 'baik',
    'okay': 'baik',
    'okelah': 'baik',
    'okeh': 'baik',
    'jelek': 'buruk',
    'jlk': 'buruk',
    'ancur': 'buruk',
    'parah': 'buruk',
    'zonk': 'buruk',
    # Code-switching umum (kata Inggris di review hotel ID)
    'bfast': 'breakfast',
    'b-fast': 'breakfast',
    'brekfas': 'breakfast',
    'breskfast': 'breakfast',
    'breakfas': 'breakfast',
    'good': 'bagus',
    'great': 'bagus',
    'nice': 'bagus',
    'bad': 'buruk',
    'helpful': 'membantu',
    'friendly': 'ramah',
    'comfortable': 'nyaman',
    'clean': 'bersih',
    'staff': 'staf',
    # Domain hotel
    'ac': 'air conditioner',
    'wifi': 'wi-fi',
    'wf': 'wi-fi',
    'tv': 'televisi',
}


def normalize_slang(text):
    """Ganti singkatan/slang dengan kata baku."""
    words = text.split()
    result = []
    for word in words:
        # Bersihkan tanda baca di akhir kata untuk matching
        clean_word = re.sub(r'[.,!?;:]+$', '', word)
        trailing = word[len(clean_word):]

        if clean_word in SLANG_DICT:
            result.append(SLANG_DICT[clean_word] + trailing)
        else:
            result.append(word)
    return ' '.join(result)


print('Menerapkan normalisasi slang...')
df['text_review'] = df['text_review'].apply(normalize_slang)
print('Selesai.')

# Sample
print('\nSample setelah normalisasi slang:')
for _, row in df.sample(5, random_state=42).iterrows():
    preview = row['text_review'][:120] + '...' if len(row['text_review']) > 120 else row['text_review']
    print(f'  {preview}')

---
## 9. Step 6: Normalisasi Karakter Berulang

Contoh: "enakkkkk" -> "enak", "baguuusss" -> "bagus"

In [ ]:
def normalize_repeated_chars(text):
    """Kurangi karakter berulang > 2 menjadi 1.
    Contoh: enakkkkk -> enak, baguuusss -> bagus"""
    return re.sub(r'(.)\1{2,}', r'\1', text)


# Cek sample sebelum
repeated = df[df['text_review'].str.contains(r'(.)\1{2,}', regex=True)]
print(f'Review dengan karakter berulang: {len(repeated):,}')
print('Sample sebelum:')
for _, row in repeated.head(5).iterrows():
    # Find the repeated part
    matches = re.findall(r'\S*(.)\1{2,}\S*', row['text_review'])
    words_with_repeat = re.findall(r'\S*(.)\1{2,}\S*', row['text_review'])
    preview = row['text_review'][:80]
    print(f'  {preview}')

# Apply
df['text_review'] = df['text_review'].apply(normalize_repeated_chars)
print(f'\nNormalisasi karakter berulang selesai.')

print('\nSample setelah:')
for _, row in df.loc[repeated.head(5).index].iterrows():
    preview = row['text_review'][:80]
    print(f'  {preview}')

---
## 10. Step 7: Hapus Tanda Baca Berlebihan

Pertahankan tanda baca dasar (. , ! ? : ;) tapi hapus yang berlebihan dan karakter dekoratif.

In [ ]:
def clean_punctuation(text):
    """Bersihkan tanda baca berlebihan tapi pertahankan yang bermakna."""
    # Hapus multiple punctuation: "!!!" -> "!", "..." -> "."
    text = re.sub(r'([!?.]){2,}', r'\1', text)

    # Hapus tanda baca dekoratif
    text = re.sub(r'[~*#@^&|\\{}\[\]<>]+', ' ', text)

    # Normalisasi spasi setelah tanda baca
    text = re.sub(r'\s*([.,!?;:])\s*', r'\1 ', text)

    # Normalisasi spasi
    text = re.sub(r' {2,}', ' ', text)
    text = text.strip()

    return text


df['text_review'] = df['text_review'].apply(clean_punctuation)
print('Normalisasi tanda baca selesai.')

---
## 11. Step 8: Final Cleaning & Near-Duplicate Check

In [ ]:
# Cek near-duplicates setelah normalisasi
before = len(df)
df = df.drop_duplicates(subset='text_review', keep='first')
removed = before - len(df)
print(f'Near-duplicates dihapus: {removed:,}')

# Final filter panjang minimal
before = len(df)
df = df[df['text_review'].str.strip().str.len() >= MIN_LENGTH]
removed = before - len(df)
print(f'Review terlalu pendek setelah cleaning: {removed:,} dihapus')

print(f'\nDataset bersih: {len(df):,} review')

---
## 12. Ringkasan Preprocessing

In [ ]:
# Reset review_id
df = df.reset_index(drop=True)
df['review_id'] = range(1, len(df) + 1)

final_count = len(df)
total_removed = initial_count - final_count

print('=' * 55)
print('  RINGKASAN PREPROCESSING')
print('=' * 55)
print(f'Review awal:     {initial_count:,}')
print(f'Review akhir:    {final_count:,}')
print(f'Total dihapus:   {total_removed:,} ({total_removed/initial_count*100:.1f}%)')

print(f'\n--- Distribusi Platform ---')
display(df.groupby('platform').size().to_frame('jumlah').reset_index())

print(f'\n--- Distribusi Hotel ---')
display(df.groupby('hotel_name').size().to_frame('jumlah').reset_index())

if 'original_language' in df.columns:
    print(f'\n--- Distribusi Bahasa Asli ---')
    display(df.groupby('original_language').size().to_frame('jumlah').reset_index())

# Statistik panjang teks
lens = df['text_review'].str.len()
print(f'\n--- Statistik Panjang Teks ---')
print(f'  Min: {lens.min()}')
print(f'  Max: {lens.max()}')
print(f'  Mean: {lens.mean():.0f}')
print(f'  Median: {lens.median():.0f}')

---
## 13. Sample Data Bersih

In [ ]:
print('Sample data bersih (10 review acak):')
print('=' * 80)
for _, row in df.sample(10, random_state=42).iterrows():
    preview = row['text_review'][:150] + '...' if len(row['text_review']) > 150 else row['text_review']
    print(f'  [{row["review_id"]}] [{row["platform"]}] [{row["hotel_name"]}]')
    print(f'  "{preview}"')
    print()

---
## 14. Export Dataset

In [ ]:
# Kolom final — pertahankan text_review_original untuk audit/transparansi
output_cols = ['review_id', 'platform', 'hotel_name',
               'text_review', 'text_review_original',
               'date', 'original_language']
# Cek apakah semua kolom ada (jaga-jaga)
output_cols = [c for c in output_cols if c in df.columns]
df_out = df[output_cols]

# Simpan ke Google Drive
output_path = os.path.join(BASE_DIR, OUTPUT_FILE)
df_out.to_csv(output_path, index=False, encoding='utf-8-sig')

file_size = os.path.getsize(output_path) / 1024 / 1024
print(f'Dataset bersih berhasil disimpan!')
print(f'  Path: {output_path}')
print(f'  Ukuran: {file_size:.2f} MB')
print(f'  Jumlah review: {len(df_out):,}')
print(f'  Kolom: {df_out.columns.tolist()}')